# Reproduction-ready AutoML experiment

**CMPE 255 — Assignment 1, Part 2**

This notebook implements a classification experiment inspired by an instructor AutoGluon tabular data-science example and organizes it with **CRISP-DM**. It deliberately contains **no claimed training output**: the restricted original dataset was unavailable, and computationally expensive AutoML training was not run.

> **How to reproduce:** place an authorized CSV at `data/dataset.csv` (or set `AUTOML_DATASET_PATH`), set `TARGET_COLUMN`, inspect the checks below, and set `RUN_AUTOML=True` only when ready. Cells use explicit readiness guards so a missing dataset or AutoGluon installation produces instructions instead of an exception.

## 1. Business Understanding

The goal is to build a reliable tabular **classification** model that predicts a user-selected target column, compare a transparent manual baseline with AutoML, and document the operational trade-offs. Success is not just a high validation score: it includes reproducibility, no target leakage, appropriate class-sensitive metrics, explainability, and a credible deployment plan.

Because neither the dataset nor its data dictionary can be distributed, the stakeholder, prediction unit, positive class, cost of false positives/negatives, and acceptance threshold **must be recorded by the reproducer before interpreting results**. The primary example metric is macro F1 (robust to class imbalance); business owners should confirm it.

## 2. Data Understanding

Configure the dataset and target below. The loader checks availability without crashing. A successful load displays only schema-oriented summaries—never invented observations. Set optional ID/leakage columns after consulting the data dictionary.

In [ ]:
from pathlib import Path
import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATASET_PATH = Path(os.getenv("AUTOML_DATASET_PATH", "data/dataset.csv"))
TARGET_COLUMN = os.getenv("TARGET_COLUMN", "target")
ID_OR_LEAKAGE_COLUMNS = []  # e.g., ["row_id", "post_outcome_field"]
TEST_SIZE = 0.20
RUN_AUTOML = False  # explicit opt-in because AutoML can be expensive
AUTOML_TIME_LIMIT = 600  # seconds; adjust to available budget

print(f"Expected dataset: {DATASET_PATH.resolve()}")
print(f"Configured target: {TARGET_COLUMN!r}")

In [ ]:
DATA_READY = False
df = None
if not DATASET_PATH.exists():
    print(
        "Dataset not found. Add an authorized CSV at "
        f"{DATASET_PATH.resolve()} or set AUTOML_DATASET_PATH. "
        "Then set TARGET_COLUMN and rerun from the top. No training was attempted."
    )
else:
    df = pd.read_csv(DATASET_PATH)
    if TARGET_COLUMN not in df.columns:
        print(
            f"Dataset loaded ({df.shape[0]} rows, {df.shape[1]} columns), but target "
            f"{TARGET_COLUMN!r} is absent. Available columns: {df.columns.tolist()}. "
            "Set TARGET_COLUMN and rerun; no training was attempted."
        )
    else:
        DATA_READY = True
        print(f"Dataset ready: {df.shape[0]} rows x {df.shape[1]} columns")
        display(df.head())
        display(df.dtypes.rename("dtype").to_frame())

## 3. Exploratory Data Analysis

The next cells inspect dimensions, duplicates, target balance, numeric distributions, and correlations. Visuals are generated only when authorized data is present; otherwise the notebook remains executable.

In [ ]:
if DATA_READY:
    print("Shape:", df.shape)
    print("Duplicate rows:", int(df.duplicated().sum()))
    display(df.describe(include="all").T)
    target_counts = df[TARGET_COLUMN].value_counts(dropna=False)
    display(target_counts.rename("count").to_frame())
    ax = target_counts.plot.bar(title="Target distribution")
    ax.set_xlabel(TARGET_COLUMN); ax.set_ylabel("Count")
    plt.tight_layout(); plt.show()
else:
    print("EDA skipped: complete the dataset configuration instructions above.")

In [ ]:
if DATA_READY:
    numeric_preview = df.drop(columns=[TARGET_COLUMN]).select_dtypes(include=np.number)
    if numeric_preview.shape[1]:
        numeric_preview.hist(figsize=(12, 8), bins=20)
        plt.suptitle("Numeric feature distributions"); plt.tight_layout(); plt.show()
        if numeric_preview.shape[1] > 1:
            corr = numeric_preview.corr(numeric_only=True)
            fig, ax = plt.subplots(figsize=(8, 6))
            image = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
            ax.set_xticks(range(len(corr)), corr.columns, rotation=90)
            ax.set_yticks(range(len(corr)), corr.columns)
            fig.colorbar(image, ax=ax, label="Correlation")
            ax.set_title("Numeric feature correlation")
            plt.tight_layout(); plt.show()
    else:
        print("No numeric predictor columns to plot.")
else:
    print("Distribution and correlation plots skipped: dataset unavailable.")

## 4. Missing-value analysis

Missingness is quantified by count and percentage. Target-missing rows cannot train a supervised model and are removed; predictor missingness is learned from **training data only** by the manual pipeline. AutoGluon handles supported missing values internally.

In [ ]:
if DATA_READY:
    missing = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": df.isna().mean().mul(100),
    }).sort_values("missing_percent", ascending=False)
    display(missing[missing["missing_count"] > 0])
    if len(missing[missing.missing_count > 0]):
        ax = missing.loc[missing.missing_count > 0, "missing_percent"].plot.bar(
            title="Missing values by column (%)"
        )
        plt.tight_layout(); plt.show()
else:
    print("Missing-value analysis skipped: dataset unavailable.")

## 5. Data cleaning

Cleaning is conservative and auditable: normalize column whitespace, convert infinities to missing values, drop duplicate rows, remove missing targets, and remove only explicitly declared ID/post-outcome fields. Domain-specific corrections should be added only with evidence from the data dictionary.

In [ ]:
clean_df = None
if DATA_READY:
    clean_df = df.copy()
    clean_df.columns = clean_df.columns.str.strip()
    if TARGET_COLUMN not in clean_df.columns:
        raise ValueError("Column trimming changed target lookup; update TARGET_COLUMN.")
    clean_df = clean_df.replace([np.inf, -np.inf], np.nan).drop_duplicates()
    clean_df = clean_df.dropna(subset=[TARGET_COLUMN])
    declared_drops = [c for c in ID_OR_LEAKAGE_COLUMNS if c in clean_df.columns and c != TARGET_COLUMN]
    clean_df = clean_df.drop(columns=declared_drops)
    print("Clean shape:", clean_df.shape, "| explicitly dropped:", declared_drops)
else:
    print("Cleaning skipped: dataset unavailable.")

## 6. Feature preparation

Features and labels are separated before modeling. For the manual baseline, numeric columns receive median imputation plus scaling; categoricals receive most-frequent imputation plus one-hot encoding with unknown categories ignored. The pipeline fits all learned transformations only on the training fold.

In [ ]:
X = y = None
if DATA_READY:
    X = clean_df.drop(columns=[TARGET_COLUMN])
    y = clean_df[TARGET_COLUMN]
    numeric_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.columns.difference(numeric_features).tolist()
    print("Numeric features:", numeric_features)
    print("Categorical features:", categorical_features)
else:
    print("Feature preparation skipped: dataset unavailable.")

## 7. Train/test split

A fixed random seed supports reproducibility. Stratification preserves class proportions when every class has at least two rows. The untouched test partition is used once for final comparison; AutoGluon's internal validation is drawn only from training data.

In [ ]:
X_train = X_test = y_train = y_test = None
if DATA_READY:
    counts = y.value_counts()
    stratify_y = y if y.nunique(dropna=True) > 1 and counts.min() >= 2 else None
    if stratify_y is None:
        warnings.warn("Split is not stratified: the target has too few observations per class.")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=stratify_y
    )
    print("Train rows:", len(X_train), "| Test rows:", len(X_test))
else:
    print("Split skipped: dataset unavailable.")

## 8. Leakage prevention

- Split before fitting imputers, scalers, encoders, feature selection, or models.
- Keep labels out of `X`; never use test data for model selection.
- Explicitly remove IDs and post-outcome fields after data-dictionary review.
- For temporal/grouped records, replace the random split with time-based or group-aware splitting.
- AutoGluon receives only `train_data` during fitting; the test frame is passed only to evaluation after training.
- Production preprocessing must be the fitted pipeline/model artifact, not a fresh fit.

## 9. Simple manual Scikit-learn baseline

A regularized logistic regression is a useful, interpretable floor. The code reports metrics only after actual fitting; no static scores are embedded.

In [ ]:
baseline = None
baseline_metrics = None
if DATA_READY:
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocess = ColumnTransformer([
        ("numeric", numeric_pipe, numeric_features),
        ("categorical", categorical_pipe, categorical_features),
    ])
    baseline = Pipeline([
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ])
    baseline.fit(X_train, y_train)
    baseline_pred = baseline.predict(X_test)
    baseline_metrics = {
        "accuracy": accuracy_score(y_test, baseline_pred),
        "precision_macro": precision_score(y_test, baseline_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, baseline_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, baseline_pred, average="macro", zero_division=0),
    }
    if y.nunique() == 2 and hasattr(baseline, "predict_proba"):
        baseline_metrics["roc_auc"] = roc_auc_score(y_test, baseline.predict_proba(X_test)[:, 1])
    display(pd.Series(baseline_metrics, name="manual_baseline"))
    print(classification_report(y_test, baseline_pred, zero_division=0))
else:
    print("Manual baseline skipped: dataset unavailable.")

## 10. AutoGluon `TabularPredictor` workflow

The import is optional and guarded. Installation instructions appear rather than an import traceback. `RUN_AUTOML` is a deliberate cost-control gate.

In [ ]:
AUTOGLUON_AVAILABLE = False
TabularPredictor = None
try:
    from autogluon.tabular import TabularPredictor
    AUTOGLUON_AVAILABLE = True
    print("AutoGluon is available.")
except ImportError:
    print(
        "AutoGluon is not installed. Install dependencies with "
        "`python -m pip install -r requirements.txt`, restart the kernel, and rerun. "
        "The remaining guarded cells will not crash."
    )

## 11. Automated model training

AutoGluon automatically trains and tunes multiple model families under the time limit. The `best_quality` preset enables stronger bagging/stacking but can be expensive. The artifact directory is local and reproducible; a seed is supplied where supported.

In [ ]:
predictor = None
autogluon_metrics = None
if DATA_READY and AUTOGLUON_AVAILABLE and RUN_AUTOML:
    train_data = X_train.copy()
    train_data[TARGET_COLUMN] = y_train.to_numpy()
    predictor = TabularPredictor(
        label=TARGET_COLUMN,
        eval_metric="f1_macro",
        path="artifacts/autogluon_predictor",
    ).fit(
        train_data=train_data,
        presets="best_quality",
        time_limit=AUTOML_TIME_LIMIT,
        ag_args_fit={"random_seed": RANDOM_STATE},
    )
    print("AutoGluon training completed from this dataset; continue to evaluation.")
elif not RUN_AUTOML:
    print("AutoML not run. Review data and costs, then set RUN_AUTOML=True and rerun this cell.")
elif not DATA_READY:
    print("AutoML not run: dataset unavailable or target not configured.")
else:
    print("AutoML not run: AutoGluon unavailable.")

## 12. AutoML leaderboard workflow

The leaderboard is generated dynamically from the held-out test frame only when a predictor exists. It is not hard-coded or represented by placeholder rankings.

In [ ]:
leaderboard = None
if predictor is not None:
    test_data = X_test.copy()
    test_data[TARGET_COLUMN] = y_test.to_numpy()
    leaderboard = predictor.leaderboard(test_data, silent=True)
    display(leaderboard)
else:
    print("No leaderboard to report because AutoGluon training was not executed in this notebook state.")

## 13. Model ensembling and stacking explanation

**Bagging** trains models on resampled folds to reduce variance. **Stacking** feeds out-of-fold predictions from diverse base learners into higher-level models, limiting direct target leakage when implemented correctly. AutoGluon's weighted ensemble combines complementary predictions and may improve robustness. These methods increase training time, memory, artifact size, and inference latency; deeper stacks are not automatically preferable. The final selection must be validated on untouched data.

## 14. Evaluation metrics

For imbalanced multiclass classification, **macro F1** gives every class equal weight and is the primary example metric. Accuracy supplies context but can hide minority-class failure. Macro precision/recall show error trade-offs; binary ROC AUC is calculated only where probability output is applicable. Confusion matrices and per-class reports support diagnosis. Business costs and threshold tuning should determine the final metric and operating point.

In [ ]:
if predictor is not None:
    ag_pred = predictor.predict(X_test)
    autogluon_metrics = {
        "accuracy": accuracy_score(y_test, ag_pred),
        "precision_macro": precision_score(y_test, ag_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, ag_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, ag_pred, average="macro", zero_division=0),
    }
    display(pd.Series(autogluon_metrics, name="autogluon"))
    display(pd.DataFrame(confusion_matrix(y_test, ag_pred),
                         index=pd.Index(sorted(y_test.unique()), name="actual"),
                         columns=pd.Index(sorted(y_test.unique()), name="predicted")))
else:
    print("AutoGluon evaluation unavailable because no predictor was trained; no values are claimed.")

## 15. Feature importance workflow

AutoGluon's permutation importance measures performance change after shuffling each feature on held-out data. It is model- and sample-dependent, can be unstable for correlated predictors, and is not causal. The workflow computes values only after training.

In [ ]:
feature_importance = None
if predictor is not None:
    feature_importance = predictor.feature_importance(test_data)
    display(feature_importance)
else:
    print("Feature importance unavailable because no AutoGluon predictor was trained; no values are claimed.")

## 16. Manual machine learning vs AutoML comparison

The following table is created only from metrics actually computed in this session. Beyond scores, compare setup effort, interpretability, artifact size, training/inference latency, and maintenance burden. Logistic regression is fast and transparent but has limited functional form; AutoML searches broader models and ensembles but consumes more resources and can be harder to audit.

In [ ]:
comparison = {}
if baseline_metrics is not None:
    comparison["Manual logistic regression"] = baseline_metrics
if autogluon_metrics is not None:
    comparison["AutoGluon"] = autogluon_metrics
if comparison:
    display(pd.DataFrame(comparison).T)
else:
    print("No measured comparison is available. Run with an authorized dataset to generate it.")

## 17. Benefits of AutoML

- Rapid, consistent exploration of diverse algorithms and hyperparameters.
- Strong tabular preprocessing defaults and fewer hand-built pipelines.
- Bagging/stacking can exploit complementary model errors.
- Reproducible leaderboard, inference, and feature-importance interfaces.
- Lets practitioners spend more time on problem framing and data quality.

## 18. Risks and disadvantages of AutoML

- Validation leakage remains possible through poor problem framing or repeated test reuse.
- Ensembles can be opaque, large, slow, and difficult to govern.
- Automated searches may amplify bias, spurious signals, or inappropriate metrics.
- Results can vary with library/hardware versions and stochastic training.
- Convenience does not replace domain review, fairness analysis, security, or monitoring.

## 19. Computational cost considerations

The time limit, presets, folds, stack levels, and hyperparameter space drive CPU/GPU time, memory, storage, energy, and cloud cost. Start with a small time budget and `medium_quality`, profile memory/latency, then justify `best_quality`. Preserve logs and versions, cap parallelism on shared machines, and compare incremental performance with incremental cost. This notebook defaults to a 600-second limit but keeps training off.

## 20. Deployment considerations

Save the complete fitted predictor and record Python/package versions, schema, label mapping, data snapshot, split seed, metric definition, and training configuration. Validate input types/ranges, reject or quarantine malformed records, secure artifacts and PII, benchmark latency/throughput, and containerize the runtime. Use shadow/canary rollout, monitor drift, class-specific quality, fairness, calibration and failures, define rollback/retraining triggers, and never train from live request data implicitly.

## 21. Limitations

The restricted dataset was not available, so schema assumptions, label semantics, imbalance, temporal/group structure, fairness, and data quality could not be verified. Training was not executed and no numerical leaderboard, ranking, feature importance, or AutoGluon performance result is reported. A random split may be invalid for time-ordered or grouped data. Macro F1 is provisional until business costs are known.

## 22. Future improvements

1. Obtain authorized data and its dictionary; define the prediction moment and leakage blacklist.
2. Replace random splitting with temporal/group-aware validation when appropriate.
3. Add domain validation, fairness slices, calibration, threshold and cost analysis.
4. Compare constrained presets/budgets and report runtime, memory, inference latency, and artifact size.
5. Add repeated or nested validation, drift checks, experiment tracking, tests, and a model card.
6. Evaluate deployment on representative hardware and define monitored retraining criteria.

## 23. CRISP-DM conclusion

This notebook covers **Business Understanding**, **Data Understanding**, **Data Preparation**, **Modeling**, **Evaluation**, and **Deployment**. It treats CRISP-DM as iterative: deployment monitoring should feed new evidence back into understanding and preparation. The complete executable workflow is present, but dataset access and explicit training opt-in are prerequisites. Consequently, this submission reports methodology—not fabricated model performance.